# 00 · Environment setup — uber-taxi-agentic-etl

One-time setup: Spark session, ADLS Gen2 access through a service principal stored in a secret scope, container/folder layout, schema creation, connectivity checks and a test call to the GPT deployment on Microsoft Foundry.

**Prerequisites** (see `infra/provision_azure.sh`): ADLS Gen2 account with containers `raw, bronze, silver, gold, quarantine, reports`; Key Vault-backed secret scope `uber-etl` holding `sp-client-id`, `sp-client-secret`, `sp-tenant-id`; a GPT deployment on Foundry.

In [ ]:
%pip install -q -r ../requirements.txt
# dbutils.library.restartPython()  # uncomment on Databricks after the first install

In [ ]:
import os, sys
from pathlib import Path

# Repo root on sys.path (works in a Databricks Git folder and locally)
ROOT = Path(os.getcwd()).resolve().parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd()).resolve()
sys.path.insert(0, str(ROOT))
ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
print("repo root:", ROOT, "| on Databricks:", ON_DATABRICKS)

## 1. Configuration (no hard-coded values — everything comes from `src/config.py`)

In [ ]:
if ON_DATABRICKS:
    dbutils.widgets.text("storage_account", "", "ADLS Gen2 account")
    dbutils.widgets.text("secret_scope", "uber-etl", "Secret scope")
    os.environ["STORAGE_MODE"] = "adls"
    os.environ["STORAGE_ACCOUNT_NAME"] = dbutils.widgets.get("storage_account")
    os.environ["DATABRICKS_SECRET_SCOPE"] = dbutils.widgets.get("secret_scope")
    os.environ.setdefault("OPTIMIZE_GOLD", "true")

from src.config import get_config
cfg = get_config()
print("storage mode:", cfg.storage_mode, "| format:", cfg.table_format)
for layer in ("raw", "bronze", "silver", "quarantine", "reports"):
    print(f"{layer:11}", cfg.container_root(layer))

## 2. Spark session and ADLS access

Secrets are read from the secret scope with `dbutils.secrets.get` — never written in code. (Skip if the cluster already has the OAuth `spark_conf` from `databricks.yml`, or if you use a Unity Catalog external location.)

In [ ]:
from src.spark_utils import get_spark, configure_adls_oauth
spark = get_spark(cfg=cfg)
if cfg.storage_mode == "adls":
    configure_adls_oauth(spark, cfg)
print("Spark", spark.version)

## 3. Create the layer folders and verify connectivity

In [ ]:
if ON_DATABRICKS:
    for layer in ("raw", "bronze", "silver", "quarantine"):
        dbutils.fs.mkdirs(f"{cfg.container_root(layer)}/{cfg.dataset}")
    for c in cfg.containers:
        files = dbutils.fs.ls(cfg.container_root(c))
        print(f"OK  {c:11} {len(files)} entries")
else:
    for layer in ("raw", "bronze", "silver", "quarantine", "gold", "reports"):
        Path(cfg.container_root(layer)).mkdir(parents=True, exist_ok=True)
    print("local lakehouse ready at", cfg.local_root)

## 4. Land the source file in `raw/uber_rides/`

Upload `ncr_ride_bookings.csv` (150k rows) to the landing zone. If it is missing, the synthetic sample from `data/sample/` is copied so the pipeline can run.

In [ ]:
from src.etl_pipeline import list_source_files
files = list_source_files(spark, cfg)
if not files:
    sample = ROOT / "data" / "sample" / "ncr_ride_bookings_sample.csv"
    if ON_DATABRICKS:
        dbutils.fs.cp(f"file:{sample}", f"{cfg.raw_path}/ncr_ride_bookings_sample.csv")
    else:
        import shutil; Path(cfg.raw_path).mkdir(parents=True, exist_ok=True); shutil.copy(sample, cfg.raw_path)
    files = list_source_files(spark, cfg)
print(files)

## 5. Databases / schemas

Register the Delta locations as tables so analysts can query them with SQL (Hive metastore shown; with Unity Catalog use `CREATE SCHEMA <catalog>.uber_etl` and external locations).

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS uber_etl")
from src.config import GOLD_TABLES
from src.spark_utils import table_exists
locations = {"bronze_rides": cfg.bronze_path, "silver_rides": cfg.silver_path, "quarantine_rides": cfg.quarantine_path,
             **{f"gold_{t}": cfg.gold_path(t) for t in GOLD_TABLES}}
if cfg.table_format == "delta":
    for name, path in locations.items():
        if table_exists(spark, path, cfg):
            spark.sql(f"CREATE TABLE IF NOT EXISTS uber_etl.{name} USING DELTA LOCATION '{path}'")
            print("registered uber_etl." + name)

## 6. Test the Foundry GPT endpoint

Authenticates with Microsoft Entra ID (`DefaultAzureCredential`) unless `FOUNDRY_AUTH=key`. Uses JSON-mode output like the agents do.

In [ ]:
from src.agents import make_llm
llm = make_llm(cfg)
print("client:", llm.name)
reply = llm.complete_json("route", "Reply with JSON only: {\"route\": \"explain\", \"reason\": \"<why>\"}",
                          {"request": "Why did yesterday's run fail?"})
print(reply, llm.usage)

Setup complete → continue with `01_pipeline_and_agents.ipynb`.